# 📊 Week 1 Task: Data Acquisition, Cleaning, and Preprocessing
**Program**: Virtual Data Science with Python Trainee | YuvaInternship  
**Author**: Aryan Kumar  
**Core Stack**: `Python 3.13` | `Pandas` | `NumPy` | `Scikit-Learn` | `Matplotlib` | `Seaborn`  

---
## 🎯 Objective & Workflow Overview
This notebook provides a complete walkthrough of acquiring a public customer intelligence dataset, diagnosing data quality issues (hidden missingness, impossible values, string formatting variance, extreme outliers), executing systematic cleaning, and engineering machine-learning-ready features.

### 🚀 Pipeline Stages:
1. **Data Acquisition & Ingestion**: Programmatic acquisition & schema identification.
2. **Data Quality Audit**: Diagnostic inspection of missingness and structural defects.
3. **Data Cleaning & Value Remediation**: Deduplication, string canonicalization, KNN imputation, and domain constraint enforcement.
4. **Outlier Diagnostics & Skewness Transformation**: Tukey 1.5x IQR Winsorization and Log1p transforms.
5. **Feature Engineering, Encoding & Scaling**: Tenure cohort binning, service density scores, One-Hot Encoding, and StandardScaler normalization.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries successfully loaded!')

## 📥 Stage 1: Data Acquisition & Inspection
We load the raw customer dataset containing real-world defects: structural missingness, inconsistent casing, currency prefixes, negative charges, and impossible tenures.

In [2]:
raw_df = pd.read_csv('raw_customer_dataset.csv')
print(f'Raw Dataset Dimensions: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
raw_df.head()

## 🔍 Stage 2: Exploratory Data Quality Auditing & Missingness Detection
We inspect both explicit `NaN` values and hidden blank strings (`' '`, `'N/A'`, `'INVALID_DATE'`).

In [3]:
missing_summary = []
for col in raw_df.columns:
    null_count = raw_df[col].isnull().sum()
    blank_count = 0
    if raw_df[col].dtype == 'object':
        blank_count = raw_df[col].apply(lambda x: 1 if isinstance(x, str) and (x.strip() == '' or x.strip().upper() in ['NA', 'N/A', 'NULL', 'NONE']) else 0).sum()
    total_missing = null_count + blank_count
    missing_summary.append({
        'Feature': col, 'Raw Nulls': null_count, 'Hidden Blanks': blank_count,
        'Total Incomplete': total_missing, 'Missing %': round((total_missing / len(raw_df)) * 100, 2)
    })
missing_df = pd.DataFrame(missing_summary).sort_values(by='Total Incomplete', ascending=False)
missing_df

## 🧹 Stage 3: Data Cleaning, Remediation & Imputation
1. **Deduplication**: Remove duplicate primary keys (`CustomerID`).
2. **String Standardization**: Case-folding and category unification.
3. **Type Parsing**: Parsing numeric features and stripping currency symbols (`$`).
4. **Domain Boundary Constraints**: Invalidate negative charges, negative tenures, and invalid satisfaction scores.
5. **Multi-Format Datetime Parsing**: Reconstruct missing/corrupted signup dates.
6. **KNN & Mode Imputation**: Distance-weighted 5-NN imputation for continuous features and mode for categoricals.

In [4]:
# Execute cleaning pipeline
from data_preprocessing_pipeline import clean_and_remediate_data
clean_df = clean_and_remediate_data(raw_df)
print(f'Cleaned Dataset Dimensions: {clean_df.shape[0]} rows, {clean_df.shape[1]} columns')
clean_df.info()

## 📈 Stage 4: Outlier Treatment & Skewness Transformation
We diagnose outliers via Tukey 1.5x IQR thresholds, perform Winsorization (robust boundary capping), and apply Log1p transformation on skewed monetary metrics.

In [5]:
from data_preprocessing_pipeline import handle_outliers_and_transformations
treated_df = handle_outliers_and_transformations(raw_df, clean_df)
treated_df[['MonthlyCharges', 'MonthlyCharges_Capped', 'TotalCharges', 'TotalCharges_Capped', 'TotalCharges_Log1p']].head()

## ⚙️ Stage 5: Feature Engineering, Encoding & Scaling
- **TenureGroup**: Binned customer lifecycle stage (`New`, `Early`, `Established`, `Loyal`).
- **TotalServicesCount**: Sum of active digital service add-ons.
- **MonthlyToTotalRatio**: Billing velocity / shock indicator.
- **One-Hot Encoding**: Nominal variables with `drop_first=True`.
- **StandardScaler**: Scaled continuous features to zero mean and unit variance.

In [6]:
from data_preprocessing_pipeline import engineer_features_and_encode
feat_df, model_ready_df = engineer_features_and_encode(treated_df)
print(f'Model-Ready Dataset Shape: {model_ready_df.shape[0]} rows, {model_ready_df.shape[1]} features')
model_ready_df.head()

## 🏁 Summary & Deliverables Export
The data pipeline has successfully generated:
- `raw_customer_dataset.csv`: Ingested raw records with data quality defects.
- `cleaned_customer_dataset.csv`: Deduplicated, remediated, and imputed records.
- `preprocessed_model_ready_dataset.csv`: 26 normalized, encoded, model-ready features.
- `Week_1_Data_Acquisition_Cleaning_Report.docx`: Complete publication-grade Word report.